# FFAI Structured Output: LiteLLM Multi-Provider

This notebook demonstrates FFAI's structured output feature across **two
LLM providers** using LiteLLM-backed clients:

- **Mistral Small** via `FFLiteLLMClient(model_string='mistral/mistral-small-2503')`
- **OpenAI GPT-4o Mini** via `FFLiteLLMClient(model_string='openai/gpt-4o-mini')`

Both clients use the **same FFAI structured output API** — pass a Pydantic
model to `response_model=` and FFAI handles schema injection, validation,
and retry automatically. LiteLLM translates the Pydantic model to each
provider's native structured output format.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from pydantic import BaseModel, Field  # noqa: E402

from ffai.Clients.FFLiteLLMClient import FFLiteLLMClient  # noqa: E402
from ffai.core.response_options import ResponseOptions  # noqa: E402
from ffai.FFAI import FFAI  # noqa: E402

In [ ]:
mistral = FFLiteLLMClient(
    model_string='mistral/mistral-small-2503',
    temperature=0.3,
    max_tokens=1024,
)

openai = FFLiteLLMClient(
    model_string='openai/gpt-4o-mini',
    temperature=0.3,
    max_tokens=1024,
)

print(f'Mistral client: {mistral}')
print(f'OpenAI client:  {openai}')

---

## Step 1: Define a Shared Pydantic Model

The same model works with both providers. FFAI passes the Pydantic class
directly to LiteLLM's `response_format` parameter, which translates it
to each provider's native schema format.

In [ ]:
class SentimentAnalysis(BaseModel):
    label: str = Field(description='One of: positive, negative, neutral')
    confidence: float = Field(description='Confidence score from 0.0 to 1.0')
    key_phrases: list[str] = Field(description='Important phrases from the text')

print(SentimentAnalysis.model_json_schema())

---

## Step 2: Sentiment Analysis with Both Providers

Run the same prompt through Mistral and OpenAI. Both return validated
`SentimentAnalysis` instances with `result.parsed`.

In [ ]:
prompt = (
    "Analyze the sentiment of this customer review: "
    "'The product quality exceeded my expectations. "
    "Shipping was fast and the packaging was excellent. "
    "Will definitely order again.'"
)

mistral_ffai = FFAI(mistral)
mistral_result = mistral_ffai.generate_response(
    prompt,
    prompt_name='sentiment_mistral',
    options=ResponseOptions(response_model=SentimentAnalysis),
)

print('=== Mistral Small ===')
print(f'Status:     {mistral_result.status}')
if mistral_result.parsed:
    print(f'Label:      {mistral_result.parsed.label}')
    print(f'Confidence: {mistral_result.parsed.confidence}')
    print(f'Phrases:    {mistral_result.parsed.key_phrases}')
else:
    print(f'Errors: {mistral_result.parsing_errors}')

In [ ]:
openai_ffai = FFAI(openai)
openai_result = openai_ffai.generate_response(
    prompt,
    prompt_name='sentiment_openai',
    options=ResponseOptions(response_model=SentimentAnalysis),
)

print('=== OpenAI GPT-4o Mini ===')
print(f'Status:     {openai_result.status}')
if openai_result.parsed:
    print(f'Label:      {openai_result.parsed.label}')
    print(f'Confidence: {openai_result.parsed.confidence}')
    print(f'Phrases:    {openai_result.parsed.key_phrases}')
else:
    print(f'Errors: {openai_result.parsing_errors}')

---

## Step 3: Nested Models

Structured output supports nested Pydantic models. Both providers
correctly generate and validate nested JSON.

In [ ]:
class Address(BaseModel):
    city: str
    country: str

class CompanyInfo(BaseModel):
    name: str
    founded: int
    headquarters: Address
    products: list[str]

prompt = 'Tell me about the company that makes the iPhone.'

mistral_result = mistral_ffai.generate_response(
    prompt,
    prompt_name='company_mistral',
    options=ResponseOptions(response_model=CompanyInfo),
)

print('=== Mistral Small ===')
if mistral_result.parsed:
    c = mistral_result.parsed
    print(f'Company:  {c.name}')
    print(f'Founded:  {c.founded}')
    print(f'HQ:       {c.headquarters.city}, {c.headquarters.country}')
    print(f'Products: {c.products}')
else:
    print(f'Errors: {mistral_result.parsing_errors}')

In [ ]:
openai_result = openai_ffai.generate_response(
    prompt,
    prompt_name='company_openai',
    options=ResponseOptions(response_model=CompanyInfo),
)

print('=== OpenAI GPT-4o Mini ===')
if openai_result.parsed:
    c = openai_result.parsed
    print(f'Company:  {c.name}')
    print(f'Founded:  {c.founded}')
    print(f'HQ:       {c.headquarters.city}, {c.headquarters.country}')
    print(f'Products: {c.products}')
else:
    print(f'Errors: {openai_result.parsing_errors}')

---

## Step 4: Constrained Fields

Pydantic `Field` constraints (`ge`, `le`, `min_length`) are enforced.
If the LLM returns an out-of-range value, FFAI retries with error feedback.

In [ ]:
class RiskAssessment(BaseModel):
    risk_score: int = Field(ge=0, le=100, description='Risk score 0-100')
    risk_level: str = Field(description='One of: low, medium, high, critical')
    factors: list[str] = Field(min_length=1, description='At least one risk factor')
    recommendation: str

prompt = (
    'Assess the risk of deploying an untested ML model '
    'to production on a Friday afternoon.'
)

mistral_result = mistral_ffai.generate_response(
    prompt,
    prompt_name='risk_mistral',
    options=ResponseOptions(response_model=RiskAssessment),
)

print('=== Mistral Small ===')
if mistral_result.parsed:
    r = mistral_result.parsed
    print(f'Score:          {r.risk_score}/100')
    print(f'Level:          {r.risk_level}')
    print(f'Factors:        {r.factors}')
    print(f'Recommendation: {r.recommendation}')
else:
    print(f'Errors: {mistral_result.parsing_errors}')

In [ ]:
openai_result = openai_ffai.generate_response(
    prompt,
    prompt_name='risk_openai',
    options=ResponseOptions(response_model=RiskAssessment),
)

print('=== OpenAI GPT-4o Mini ===')
if openai_result.parsed:
    r = openai_result.parsed
    print(f'Score:          {r.risk_score}/100')
    print(f'Level:          {r.risk_level}')
    print(f'Factors:        {r.factors}')
    print(f'Recommendation: {r.recommendation}')
else:
    print(f'Errors: {openai_result.parsing_errors}')

---

## Step 5: Side-by-Side Comparison

Run the same extraction task through both providers and compare results.

In [ ]:
class MovieReview(BaseModel):
    title: str
    rating: int = Field(ge=1, le=10)
    genre: list[str]
    verdict: str

prompt = 'Review the movie "The Matrix" and rate it.'

results = {}
for label, client in [('Mistral', mistral), ('OpenAI', openai)]:
    ffai = FFAI(client)
    result = ffai.generate_response(
        prompt,
        prompt_name=f'review_{label.lower()}',
        options=ResponseOptions(response_model=MovieReview),
    )
    results[label] = result

for label, result in results.items():
    print(f'=== {label} ===')
    if result.parsed:
        r = result.parsed
        print(f'Title:   {r.title}')
        print(f'Rating:  {r.rating}/10')
        print(f'Genre:   {r.genre}')
        print(f'Verdict: {r.verdict}')
    else:
        print(f'Errors:  {result.parsing_errors}')
    print()

---

## Step 6: Usage and Cost Tracking

Both providers report token usage and estimated cost.

In [ ]:
print('=== Mistral Usage ===')
if mistral.last_usage:
    print(f'Input tokens:  {mistral.last_usage.input_tokens}')
    print(f'Output tokens: {mistral.last_usage.output_tokens}')
    print(f'Cost:          ${mistral.last_cost_usd:.6f}')

print()

print('=== OpenAI Usage ===')
if openai.last_usage:
    print(f'Input tokens:  {openai.last_usage.input_tokens}')
    print(f'Output tokens: {openai.last_usage.output_tokens}')
    print(f'Cost:          ${openai.last_cost_usd:.6f}')

---

## Summary

| Feature | How to use |
|---------|------------|
| Structured output | `options=ResponseOptions(response_model=MyPydanticModel)` |
| Access result | `result.parsed` — validated Pydantic instance |
| Validation errors | `result.parsing_errors` — list of error strings |
| Auto-retry | On failure, FFAI retries up to 3 times with error feedback |
| Multi-provider | Same Pydantic model works with any LiteLLM client |

**How LiteLLM handles structured output:**

1. FFAI passes the Pydantic model class directly to `litellm.completion(response_format=)`
2. LiteLLM translates the model to each provider's native format:
   - **OpenAI**: `json_schema` response format
   - **Mistral**: `json_object` response format with schema in system prompt
3. The LLM generates JSON matching the schema
4. FFAI validates with Pydantic and retries on failure